In [1]:
import torch
from requests.packages import target

In [2]:
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

# here are all the unique characters that occur in this text

In [3]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print("All the unique characters:", ''.join(chars))
print("Vocabulary size:", vocab_size)

All the unique characters: 
 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
Vocabulary size: 65


# create a mapping from characters to integers

In [4]:
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda text_token :[stoi[c] for c in text_token]
decode = lambda int_token :''.join([itos[i] for i in int_token])
print(encode("hii there"))
print(decode(encode("hii there")))

[46, 47, 47, 1, 58, 46, 43, 56, 43]
hii there


# let's now encode the entire text dataset and store it into a torch.Tensor

In [5]:
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:1000])

torch.Size([1115389]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1, 42, 47, 43,  1, 58, 46, 39, 52,  1, 58, 53,  1, 44, 39, 51, 47,
        57, 46, 12,  0,  0, 13, 50, 50, 10,  0, 30, 43, 57, 53, 50, 60, 43, 42,
         8,  1, 56, 43, 57, 53, 50, 60, 43, 42,  8,  0,  0, 18, 47, 56, 57, 58,
         1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 18, 47, 56, 57, 58,  6,  1, 63,
        53, 59,  1, 49, 52, 53, 61,  1, 15, 39, 47, 59, 57,  1, 25, 39, 56, 41,
      

# Split data into train and validation

In [6]:
train_val_data_percentage = int(0.9 * len(data))
train_data = data[:train_val_data_percentage]
validation_data = data[train_val_data_percentage:]

In [7]:
block_size = 8
train_data[:block_size+1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [8]:
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]
    reference = y[t]
    print(f"when input {context} and output {reference}")


when input tensor([18]) and output 47
when input tensor([18, 47]) and output 56
when input tensor([18, 47, 56]) and output 57
when input tensor([18, 47, 56, 57]) and output 58
when input tensor([18, 47, 56, 57, 58]) and output 1
when input tensor([18, 47, 56, 57, 58,  1]) and output 15
when input tensor([18, 47, 56, 57, 58,  1, 15]) and output 47
when input tensor([18, 47, 56, 57, 58,  1, 15, 47]) and output 58


In [9]:
torch.manual_seed(1337)
batch_size = 8 # how many independent sequences will we process in parallel?
block_size = 4 # what is the maximum context length for predictions?

# generate a small batch of data of inputs x and targets y
def get_batch(split):
    data = train_data if split == 'train' else validation_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

for b in range(batch_size): # batch dimension
    for t in range(block_size): # time dimension
        context = xb[b, :t+1]
        target = yb[b,t]
        print(f"when input is {context.tolist()} the target: {target}")

inputs:
torch.Size([8, 4])
tensor([[57,  1, 46, 43],
        [58, 46, 39, 58],
        [46, 39, 58,  1],
        [ 0, 21,  1, 54],
        [52,  1, 63, 43],
        [ 0, 25, 63,  1],
        [ 1, 57, 39, 49],
        [ 6,  1, 58, 46]])
targets:
torch.Size([8, 4])
tensor([[ 1, 46, 43, 39],
        [46, 39, 58,  1],
        [39, 58,  1, 46],
        [21,  1, 54, 39],
        [ 1, 63, 43, 39],
        [25, 63,  1, 45],
        [57, 39, 49, 43],
        [ 1, 58, 46, 47]])
when input is [57] the target: 1
when input is [57, 1] the target: 46
when input is [57, 1, 46] the target: 43
when input is [57, 1, 46, 43] the target: 39
when input is [58] the target: 46
when input is [58, 46] the target: 39
when input is [58, 46, 39] the target: 58
when input is [58, 46, 39, 58] the target: 1
when input is [46] the target: 39
when input is [46, 39] the target: 58
when input is [46, 39, 58] the target: 1
when input is [46, 39, 58, 1] the target: 46
when input is [0] the target: 21
when input is [0, 21]

In [10]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)
# (B, T, C)
# B-Batch
# T-Time
# C-Channel
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        # idx and targets are both (B,T) tensor of integers
        logits = self.token_embedding_table(idx) # (B,T,C)
        if target is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self(idx)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)

print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=100)[0].tolist()))

torch.Size([32, 65])
tensor(4.6847, grad_fn=<NllLossBackward0>)


AttributeError: 'NoneType' object has no attribute 'view'